#这个环境有scFates, 有cellrank和scfates，有试过官方推荐的（一种依赖于细胞的命运概率图，使用 Palantir（Setty 等人，2019 年）和 CellRank（Lange 等人，2022 年）等工具。），Palantir效果很差所以不采用，用cellrank模式从头开始。

In [ ]:
import cellrank as cr
import scanpy as sc

sc.settings.set_figure_params(frameon=False, dpi=100)
cr.settings.verbosity = 2

In [ ]:
import warnings

warnings.simplefilter("ignore", category=UserWarning)

##cellrank部分

In [ ]:
adata = sc.read_h5ad("data/ibd_T_only_dpt.h5ad")

In [ ]:
adata

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color=["clusters", "dpt_pseudotime"])

In [ ]:
from cellrank.kernels import PseudotimeKernel

pk = PseudotimeKernel(adata, time_key="dpt_pseudotime")

In [ ]:
pk

In [ ]:
pk.compute_transition_matrix()

In [ ]:
pk

In [ ]:
pk.plot_random_walks(
    seed=0,
    n_sims=100,
    start_ixs={"clusters": 'Naive_T'},
    basis="X_umap",
    legend_loc="right",
    dpi=150,
)

In [ ]:
from cellrank.kernels import ConnectivityKernel

ck = ConnectivityKernel(adata).compute_transition_matrix(density_normalize=True)

In [ ]:
combined_kernel = 0.8 * pk + 0.2 * ck

In [ ]:
from cellrank.estimators import GPCCA
g = GPCCA(combined_kernel)
print(g)

In [ ]:
g.fit(n_states=6, cluster_key="clusters")
g.plot_macrostates(which="all")

In [ ]:
g.predict_initial_states()
g.plot_macrostates(which="initial", s=100)

In [ ]:
g.predict_terminal_states(method="top_n", n_states=5, allow_overlap=True)
g.plot_macrostates(which="terminal")


In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(legend_loc="right")

In [ ]:
cr.pl.circular_projection(adata, keys="clusters", legend_loc="right")

In [ ]:
mono_drivers = g.compute_lineage_drivers(lineages="Treg")

In [ ]:
mono_drivers = g.compute_lineage_drivers(lineages="TH17")
genename = 'gene_name'  # 定义列名（对应R中的genename变量）
mono_drivers[genename] = mono_drivers.index
mono_drivers.head(10)

In [ ]:
mono_drivers.Treg_corr

In [ ]:
model = cr.models.GAMR(adata)

In [ ]:
adata.obs["inflammation_score_num"] = (
    adata.obs["inflammation_score"]
    .str.extract(r"([0-9.]+)$")[0]
    .astype(float)
)

In [ ]:
cr.pl.gene_trends(
    adata,
    model=model,
    data_key="scaled",
    genes=["BATF", "FOXP3",'IL17A','CXCR5','STAT3'],
    same_plot=True,
    ncols=2,
    time_key="dpt_pseudotime",
    hide_cells=True,
)

In [ ]:
import warnings
warnings.simplefilter("ignore", category = UserWarning)
warnings.simplefilter("ignore", category = FutureWarning)
warnings.simplefilter("ignore", category = DeprecationWarning)

换环境为scfates的环境

In [ ]:
import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np
import os, sys
from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import scFates as scf
import palantir
import matplotlib.pyplot as plt
import os, sys
os.environ['R_HOME'] = sys.exec_prefix+"/lib/R/"

scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo')
cr.settings.verbosity = 2

In [ ]:
sc.pl.umap(adata,color=['dpt_pseudotime', 'clusters'],cmap='RdBu_r')

In [ ]:
from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import scFates as scf
import palantir
import matplotlib.pyplot as plt
import os, sys

In [ ]:
scf.tl.cellrank_to_tree(adata,time="dpt_pseudotime",Nodes=300,seed=1)

In [ ]:
sig=scf.tl.explore_sigma(adata,Nodes=50,use_rep="X_umap",seed=33,plot=True)

In [ ]:
adata

###调参数到一个想要的路径信息

In [ ]:
scf.tl.tree(adata,
            Nodes=30,
            use_rep="X_umap",
            method="ppt",
            ppt_nsteps=5,
            ppt_sigma=1,
            ppt_lambda=10

            ,
            seed=56

            ,plot=True)

###也可以直接秒一个参数


In [ ]:
scf.tl.root(adata,20)
scf.tl.pseudotime(adata)

In [ ]:
sc.pl.umap(adata,color=["seg","milestones"])

In [ ]:
scf.pl.trajectory(adata,color_cells="milestones",basis='umap')

In [ ]:
scf.tl.dendrogram(adata)

In [ ]:
scf.pl.milestones(adata,annotate=True,basis='umap' )

In [ ]:

scf.pl.dendrogram(adata,color_milestones=True,color="milestones")
scf.pl.dendrogram(adata,color="clusters")

In [ ]:
scf.tl.test_association(adata,n_jobs=10)

In [ ]:

scf.pl.test_association(adata)

In [ ]:
scf.tl.fit(adata,n_jobs=20)

In [ ]:
scf.pl.dendrogram(adata,color=["clusters"],legend_loc="on data",color_milestones=True,legend_fontoutline=True)
scf.pl.dendrogram(adata,color=["milestones"],legend_loc="on data",color_milestones=True,legend_fontoutline=True)

In [ ]:
scf.tl.test_fork(adata,root_milestone="20",milestones=['6','0','16','24','28'],n_jobs=20,rescale=True)

In [ ]:
scf.tl.branch_specific(
    adata,
    root_milestone="20",
    milestones=['6','0','16','24','28'],
    effect=1       # try 1 first; see note below

)

In [ ]:
import matplotlib.pyplot as plt

# 调用趋势图绘制函数
g1=scf.pl.trends(
    adata,
    root_milestone="20",
    milestones=['6','0','16','24','28'],
    branch="6",
    plot_emb=False,
    ordering="max",
    return_genes=True
)


In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

# 调用趋势图绘制函数
g2=scf.pl.trends(
    adata,
    root_milestone="20",
    milestones=['6','0','16','24','28'],
    branch="0",
    plot_emb=False,
    ordering="max",
    return_genes=True
)


In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

# 调用趋势图绘制函数
g3=scf.pl.trends(
    adata,
    root_milestone="20",
    milestones=['6','0','16','24','28'],
    branch="24",
    plot_emb=False,
    ordering="max",
    return_genes=True
)


In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

# 调用趋势图绘制函数
g4=scf.pl.trends(
    adata,
    root_milestone="20",
    milestones=['6','0','16','24','28'],
    branch="28",
    plot_emb=False,
    ordering="max",
    return_genes=True
)

In [ ]:
gg=g1.tolist()+g2.tolist()+g3.tolist()+g4.tolist()

In [ ]:
import matplotlib.pyplot as plt
g=scf.pl.trends(adata,gg,figsize=(4,4),annot="milestones",n_features=8,
        plot_emb=False,ordering=None,return_genes=True)

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# 例如：
plt.rcParams['figure.dpi'] = 100          # notebook 里的分辨率
plt.rcParams['savefig.dpi'] = 300         # 保存图的分辨率
plt.rcParams['figure.figsize'] = (4, 4)   # 默认尺寸
plt.rcParams['font.size'] = 10            # 字体大小

# 后面照常画图
scf.pl.matrix(adata, gg, norm="minmax", cmap="RdBu_r", colorbar=False)


In [ ]:
sc.set_figure_params()
scf.pl.matrix(adata,gg,norm="minmax",cmap="RdBu_r",colorbar=False,save="_F.pdf")

In [ ]:
scf.tl.activation_lm(adata,root_milestone="34",milestones=["28",'5'],n_jobs=10)

In [ ]:
scf.pl.modules(adata,root_milestone="34",milestones=["5",'28'],
               smooth=True,module="early")

In [ ]:
scf.tl.slide_cells(adata,root_milestone="22",milestones=["5",'36'],win=700)

In [ ]:
scf.tl.slide_cors(adata,root_milestone="22",milestones=["5","36"])

In [ ]:
sc.set_figure_params()
scf.pl.slide_cors(adata,root_milestone="22",milestones=["5","36"],basis="umap")

In [ ]:
print(adata.uns.keys())

In [ ]:
scf.tl.synchro_path(adata,root_milestone="16",milestones=["5","36"],w=100,n_map=20,n_jobs=20)

In [ ]:
scf.pl.synchro_path(adata,root_milestone="16",milestones=["5","36"],save="_I.pdf")

In [ ]:
scf.tl.module_inclusion(adata,root_milestone="16",milestones=["5","36"],n_jobs=20,n_map=20,parallel_mode="mappings")

In [ ]:
sc.set_figure_params(fontsize=10)
scf.pl.module_inclusion(adata,root_milestone="16",milestones=["5","36"],
                        bins=10,branch="5",save="_J1.pdf",figsize=(2,5))

In [ ]:
scf.pl.module_inclusion(adata,root_milestone="16",milestones=["5","36"],bins=10,branch="36",
                       save="_J2.pdf",figsize=(2,4))

In [ ]:
adata.write('CD8T_only_dpt_cellrank_finish.h5ad')